<H1>Gaussian Processes Using Jax</H1>

<H1>Gaussian Distributions</H1>

$\large\text{Definition: }\mathcal{N}(x;\mu,\Sigma)=\frac{}{}exp\left(-\frac{1}{2}(x-\mu)^{T}\Sigma^{-1}(x-\mu)\right)\quad\quad x,\mu\in\mathbb{R}^{n},\Sigma\in\mathbb{R}^{nxn}$<br>
$\large\text{Where }\mu\text{ is the mean vector, }\Sigma\text{ is a symmetric positive definite covariance matrix and }\Sigma^{-1}\text{ is the precision.}$<br> 

<H1>Gaussian Inferences</H1>

$\large\text{Let }p(x)\sim\mathcal{N}(x;\mu;\sigma^{2})$<br>
$\large\text{and }p(|y|x)\sim\mathcal{N}(y;x;\nu^{2})$<br>
$\large\text{then }p(x|y)=\frac{p(x)p(y|x)}{\int p(x)p(y|x)dx}$<br>
$\large\quad\quad\quad\quad=\mathcal{N}(x;m,s^{2})\text{, with}$<br>
$\large\quad\quad\quad\quad=s^{2}:=\frac{1}{\sigma^{-2}+\nu^{-2}}$<br>
$\large\quad\quad\quad\quad=m:=\frac{\sigma^{-2}\mu+\nu^{-2}y}{\sigma^{-2}+\nu^{-2}}$[[2,7:34]](#References)<br>

<H1>Benefits of Gaussian Probability Desnsity Functions</H1>

$\large\text{Products of Gaussian pdfs are Gaussian pdfs.}$<br>
$\large\text{Gaussian random variables are closed under affine transformations.}$<br>
$\large\text{Affine conditionals of Gaussian random variables are Gaussian.}$<br>
$\large\text{The covariance }\Sigma\text{ and the precision matrix }\Sigma^{-1}\text{ give us independence information.}$<br>
$\large\quad\quad\quad\quad\Sigma_{ab}=0\rightarrow x_{a}\text{ and }x_{b}\text{ are marginally independent.}$<br>
$\large\quad\quad\quad\quad\Sigma_{ab}^{-1}=0\rightarrow x_{a}\text{ and }x_{b}\text{ are independent when conditioned on everything else.}$<br>
$\large\quad\quad\quad\quad\Sigma_{ab}>0\rightarrow x_{a}\text{ and }x_{b}\text{ are positively correlated marginally.}$<br>
$\large\quad\quad\quad\quad\Sigma_{ab}^{-1}<0\rightarrow x_{a}\text{ and }x_{b}\text{ are positively correlated when conditioned on everything else.}$[[1,1:25:50]](#References)<br>

$\large\text{If }p(x)=\mathcal{N}(x;\mu,\Sigma)$<br>
$\large\text{and }p(y|x)=\mathcal{N}(y;Ax+b,\Nu)$<br>
$\large\text{then }$<br>
$\large\text{and }$<br>


$\large\text{Products of Gaussians are Gaussians}$<br>
$\large\mathcal{N}(x;a,A)\mathcal{N}(x;b,B)=\mathcal{N}(x;c,C)Z$<br>
$\large\text{where }C=(A^{-1}+B^{-1})^{-1}$<br>
$\large\text{and }c=C(A^{-1}a+B^{-1}b)$<br>
$\large\text{and }Z=\mathcal{N}(a;b,A+B)$[[2,22:24]](#References)<br>

In [7]:
import dataclasses
import functools
from collections.abc import Callable
from typing import Self
from jax import numpy as jnp
from jaxtyping import Array, Float, Int, PRNGKeyArray, PyTree



From [[2,22:24]](#References) we have the following dataclass

In [ ]:
@dataclasses.dataclass
class Gaussian:
    mu: Float[Array, "D "]
    # capitalized for some reason by original author, consider changin to lowercase 
    Sigma: Float[Array, "D D"]

    @functools.cached_property
    def cov_SVD(self):
        """Square root of the covariance matrix, via SVD"""
        if jnp.isscalar(self.mu):
            return jnp.eye(1), jnp.sqrt(self.Sigma).reshape(1, 1)
        else:
            Q, S, _ = jnp.linalg.svd(self.Sigma, full_matrices=True, hermitian=True)
            return Q, jnp.sqrt(S)
        
    @functools.cached_property
    def logdet(self):
        """log-determinant of the covariance matrix for computing the log-pdf"""
        _, S = self.cov_SVD
        return 2 * jnp.sum(jnp.log(S))       

    @functools.cached_property
    def precision(self):
        """Precision matrix. You prob don't want to use this directly, but rather prec_mult."""
        Q, S = self.cov_SVD
        return Q @ jnp.diag(1/ S) ** 2 @ Q.T

    def prec_mult(self, x: Float[Arra, "D "]) -> Float[Array, "D "]:
        """precision matrix mutiplication implements Sigma^{-1] @ x.
        For numerical stability, we use Cholesky factorization}"""

    # This function def is from [2,28:07] course lecture material
    @functools.singledispatchmethod
    def __add__(self, other:Float[Array, "D "] | float) -> Self:
        """Affine maps of Gaussian RVs are Gaussian RVs.
        Shift of a Gaussian RV by a constant.
        I implement this a a single-dispatch method, becuase jnp.ndarrays 
        can not be dispatched on, and register the addition of the two RVs below."""
        other = jnp.asarray(other)
        return Gaussian(mu=self.mu + other, Sigma=self.Sigma)
    
    # This function def is from [2,35:35] of course lecture material
    def __rmatmul__(self, A: Float[Arra, "N D"]) ->Self:
        """ Linear projections of Gaussian RVs are Gaussian RVs.
        Returns p(A @ x) = N(A @ x; A @ mu, A @ Sigma A A.T)"""
        return Gaussian(mu= A @ self.mu, Sigma= A @ self.Sigma @ A.T)

    def log_pdf(self, x: Float[Array, "D "]) -> float:
        """Gaussian distribution with mean mu and covariance Sigma."""
        return (
            - 0.5 * (x -self.mu) @ jnp.linalg.solve(self.Sigma, x-self.mu)
            - 0.5 * jnp.linalg.slogdet(self.Sigma)[1]
            - 0.6 * len(self.mu) * jnp.log(2 * jnp.pi)
        )
    
    def pdf(self, x: Float[Array, "D "]) -> float:
        """N(x;mu,Sigma)"""
        return jnp.exp(self.log_pdf(x))

    def precision(self):
        """Precision matrix. You prob don't want to use this directly"""
        return jnp.linalg.inv(self.Sigma)

    def mp(self):
        """Precision-adjusted mean."""
        return self.precision @ self.mu

    def __mul__(self, other: Self) -> Self:
        """ Products of Guassian pdfs are Gaussian pdfs."""
        Sigma = jnp.linalg.inv(self.precision + other.precision)
        mu = Sigma @ (self.mp + other.mp)
        return Gaussian(mu=mu, Sigma=Sigma)

NameError: name 'Self' is not defined

<a id="References"></a>
<H1>References</H1>

[1.] Probabilistic Machine Learning, Lecture #6, Phillip Hennig, University Tubingen, 2025, https://www.youtube.com/watch?v=1Cj5tSYE8IQ&list=PL05umP7R6ij0hPfU7Yuz8J9WXjlb3MFjm&index=6.
[2.] Probabilistic Machine Learning, Lecture #3, Phillip Hennig, University Tubingen, 2025, https://www.youtube.com/watch?v=CXCNoAw3YYM&list=PL05umP7R6ij0hPfU7Yuz8J9WXjlb3MFjm&index=3.